# 02 - Paper classical machine-learning models

This notebook reconstructs the classical supervised classification section around the **80,207-row / 218-label** classification pipeline.

The kNN configuration produces **98.609899% held-out accuracy**, matching the paper's reported 98.6% result.

The classification task uses 218 labels.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.append("..")
from src.labels import fine_label_ids_in_original_order

DATA_PATH = Path("../data/new_features_3sec.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "The recovered 80,207-row new_features_3sec.csv is missing. "
        "See data/README.md and notebook 01 to regenerate it from the original audio."
    )

data = pd.read_csv(DATA_PATH)
y, mapping = fine_label_ids_in_original_order(data["filename"])
X_df = data.drop(columns=["filename", "length"])

print("Table shape:", data.shape)
print("Numeric features:", X_df.shape[1])
print("Fine-grained labels:", len(mapping))

In [ ]:
# Historical preprocessing preserved from the recovered notebook:
# MinMaxScaler was fit before the split and no random_state was stored.
scaler = MinMaxScaler()
X = scaler.fit_transform(X_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
from sklearn.metrics import accuracy_score
from src.classical import (
    paper_knn,
    paper_logistic_regression,
    paper_sgd,
    paper_random_forest,
    paper_svc,
    paper_xgb,
    paper_xgbrf,
)

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    score = accuracy_score(y_test, pred)
    print(f"{name}: {score:.6%}")
    return score

## kNN - strongest surviving classical result

Paper configuration:

`n_neighbors=2, p=1, weights='distance'`

Surviving historical output:

`0.986098990150854`

In [ ]:
knn = paper_knn()
# score = evaluate("kNN", knn)
# Historical recovered score: 0.986098990150854

## Other paper models

These constructors use the parameters reported in Table 1.2.

They are left unexecuted because several are expensive on 80k samples. The published Table 1.3 values are documented in `RESULTS.md`.

In [ ]:
models = {
    "Logistic Regression": paper_logistic_regression(),
    "SGD": paper_sgd(),
    "Random Forest": paper_random_forest(),
    "SVC": paper_svc(),
    "XGB": paper_xgb(),
    "XGBRF": paper_xgbrf(),
}

models